In [ ]:
# Abstraction Steering (Search ver.)
# ----------------------------------
import time, os, json
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader
from transformers import AutoModelForCausalLM, AutoTokenizer
from sorl.steer import StackedAbstractionWrapperV9
from data.pt_dataset import get_dataset

# ── Config ──
MODEL_NAME = "Qwen/Qwen3-0.6B"
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
DATASET = "scienceqa"
MAX_LENGTH = 512
BATCH_SIZE = 2
GRAD_ACCUM = 4
NUM_EPOCHS = 1
LR = 1e-5
STEER_LR = 5e-2
WARMUP_STEPS = 50
MAX_GRAD_NORM = 1.0
LOG_EVERY = 10
C_SIZE = 4
L_CHUNK = 4
SCALE = 0.5
INJECT_LAYERS = [14]
NUM_ROLLOUTS = 4

# ── Load model + tokenizer ──
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
if tokenizer.pad_token_id is None:
    tokenizer.pad_token_id = tokenizer.eos_token_id
model = AutoModelForCausalLM.from_pretrained(MODEL_NAME)
D_MODEL = model.config.hidden_size
N_LAYERS = model.config.num_hidden_layers
model = model.to(DEVICE)
print(f"Model: {MODEL_NAME} | D={D_MODEL} | layers={N_LAYERS}")

# ── Build V9 wrapper ──
wrapper_v9 = StackedAbstractionWrapperV9(
    model, C_SIZE=C_SIZE, D_MODEL=D_MODEL,
    inject_layers=INJECT_LAYERS, scale=SCALE, L=L_CHUNK,
    code_position="first",
)
wrapper_v9.abs_proj.to(DEVICE)
wrapper_v9.steering_emb.to(DEVICE)

steer_params = wrapper_v9.get_steer_params()
n_steer = sum(p.numel() for p in steer_params)
n_model = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"V9: C={C_SIZE}, L={L_CHUNK}, scale={SCALE}, layers={INJECT_LAYERS}")
print(f"Trainable: model={n_model/1e6:.1f}M  steer={n_steer/1e3:.1f}K (abs_proj + steering_emb)")

# ── Dataset ──
train_ds = get_dataset(DATASET, split="train", tokenizer=tokenizer, max_length=MAX_LENGTH)
print(f"Train: {len(train_ds)} samples")

/Users/fangyuanyu/anaconda3/lib/python3.11/site-packages/torch/utils/_pytree.py:185: FutureWarning: optree is installed but the version is too old to support PyTorch Dynamo in C++ pytree. C++ pytree support is disabled. Please consider upgrading optree using `python3 -m pip install --upgrade 'optree>=0.13.0'`.
  warnings.warn(


Loading weights:   0%|          | 0/311 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie model.embed_tokens.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


Model: Qwen/Qwen3-0.6B | D=1024 | layers=28


In [ ]:
# Hyp 1. Initialization for steering emb matters, for STE
#        - test out 
# Hyp 2. Including "tempted sampling" during training to improve performance
#        - in v6 wrapper, use ".routing_temperature=1.0" to test it out, just comapre it againt temp=0.0 in same condition
#        - gsm8k + sciqa (q06) suffices
# Hyp 3. Including "search" to improve abstraction routing beyond being static, adopting v1 methods
#        - abs_route(h.detach()) -> do not joint training policy & rep (_detach_routing=True in wrapper v9)
#        - abs_route(h) -> joint training policy & rep (_detach_routing = False in wrapper v9)


In [ ]:
# ── SoRL-style Training Loop for V9 ─────────────────────────────────────
# Phase per batch:
#   1. Base CE loss (no steering, scale=0) — the "SFT baseline" for this batch
#   2. Search: repeat batch N times, forward(temperature=T, reduction='none'),
#      each copy samples different codes → select best per original sample
#   3. Train: forward(forced_codes=best_codes) — exact winning routing
#      - ce_loss:   standard next-token CE (with winning steering)
#      - info_gain: CE(steered) - CE(base) — negative means steering helped
#      - abs_loss:  CE(abs_proj logits, best_codes) — teach routing to reproduce search result
#      - zipf_loss: encourage diverse 2-gram usage of abstraction codes

from collections import defaultdict
from sorl.sorl_trainer import VariableZipfian2gramLoss as ZipLoss

ALPHA_INFO = 1.0
ALPHA_ABS  = 0.5
ALPHA_ZIPF = 0.01
SEARCH_TEMP = 1.0

zipf_loss_fn = ZipLoss(vocab_size=torch.tensor(C_SIZE, device=DEVICE))

def collate_fn(batch):
    out = {}
    for k in batch[0]:
        vals = [b[k] for b in batch]
        if isinstance(vals[0], torch.Tensor):
            out[k] = torch.stack(vals)
        else:
            out[k] = torch.tensor(vals)
    return out

dataloader = DataLoader(
    train_ds, batch_size=BATCH_SIZE, shuffle=True,
    collate_fn=collate_fn, num_workers=0, pin_memory=False,
)
total_steps = len(dataloader) * NUM_EPOCHS // GRAD_ACCUM

param_groups = [
    {'params': [p for p in model.parameters() if p.requires_grad], 'lr': LR},
    {'params': steer_params, 'lr': STEER_LR},
]
optimizer = torch.optim.AdamW(param_groups, weight_decay=0.01)

def get_lr(step, total, warmup, base_lr):
    if step < warmup:
        return base_lr * step / max(warmup, 1)
    frac = (step - warmup) / max(total - warmup, 1)
    return base_lr * 0.5 * (1 + torch.cos(torch.tensor(frac * 3.14159)).item())

# ── Training ──
history = defaultdict(list)
model.train()
global_step = 0
t_start = time.time()

for epoch in range(NUM_EPOCHS):
    for batch_idx, batch in enumerate(dataloader):
        input_ids = batch['input_ids'].to(DEVICE)
        attn = batch['attention_mask'].to(DEVICE)
        prompt_len = batch['prompt_len'].to(DEVICE)
        B = input_ids.size(0)

        # Labels: mask padding + prompt
        labels = input_ids.clone()
        labels[attn == 0] = -100
        si = torch.arange(labels.size(1), device=DEVICE).unsqueeze(0)
        labels[si < prompt_len.unsqueeze(1)] = -100

        # ── Phase 1: Base CE (no steering) ──
        old_scale = wrapper_v9.scale
        wrapper_v9.scale = 0.0
        with torch.no_grad():
            base_out = wrapper_v9(input_ids, attn, labels)
            base_ce = base_out.loss.item()
        wrapper_v9.scale = old_scale

        # ── Phase 2: Search — parallel rollouts via temperature sampling ──
        N = NUM_ROLLOUTS
        rep_ids = input_ids.repeat_interleave(N, dim=0)    # (B*N, S)
        rep_attn = attn.repeat_interleave(N, dim=0)
        rep_labels = labels.repeat_interleave(N, dim=0)

        with torch.no_grad():
            rep_out = wrapper_v9(rep_ids, rep_attn, rep_labels,
                                temperature=SEARCH_TEMP, reduction='none')

            # Per-sample loss → (B, N) → pick best rollout per sample
            per_sample_loss = rep_out.per_sample_loss.view(B, N)
            best_idx = per_sample_loss.argmin(dim=-1)  # (B,)

            # Extract winning codes
            all_codes = wrapper_v9._last_chunk_codes.view(B, N, -1)
            best_codes = all_codes[torch.arange(B, device=DEVICE), best_idx]

            del rep_out

        # ── Phase 3: Train with forced_codes = search winners ──
        out = wrapper_v9(input_ids, attn, labels, forced_codes=best_codes)
        ce_loss = out.loss

        info_gain = ce_loss - base_ce # information gain
        abs_loss_val = wrapper_v9.compute_abs_loss(target_codes=best_codes) # memorize good abstraction choice
        # Zipf 2-gram: flatten routing logits to (B*n_chunks, C) for diversity loss
        routing_logits_flat = wrapper_v9._last_routing_logits.reshape(-1, C_SIZE)
        zipf_loss = zipf_loss_fn(routing_logits_flat) # route to diverse abstractions

        loss = ce_loss + ALPHA_INFO * info_gain + ALPHA_ABS * abs_loss_val + ALPHA_ZIPF * zipf_loss
        loss = loss / GRAD_ACCUM
        loss.backward()

        if (batch_idx + 1) % GRAD_ACCUM == 0:
            if MAX_GRAD_NORM > 0:
                all_params = list(model.parameters()) + steer_params
                torch.nn.utils.clip_grad_norm_(all_params, MAX_GRAD_NORM)
            optimizer.step()
            optimizer.zero_grad(set_to_none=True)
            global_step += 1

            lr = get_lr(global_step, total_steps, WARMUP_STEPS, LR)
            optimizer.param_groups[0]['lr'] = lr
            optimizer.param_groups[1]['lr'] = lr * (STEER_LR / max(LR, 1e-10))

        # Logging
        total_loss = loss.item() * GRAD_ACCUM
        if (batch_idx + 1) % LOG_EVERY == 0:
            steer_norm = sum(p.float().norm(dim=-1).mean().item()
                             for p in wrapper_v9.steering_emb.parameters())
            abs_l = abs_loss_val.item()
            zl = zipf_loss.item()
            ig = info_gain.item() if torch.is_tensor(info_gain) else info_gain
            print(f"ep {epoch + (batch_idx+1)/len(dataloader):.3f} | "
                  f"step {global_step}/{total_steps} | "
                  f"loss={total_loss:.4f} base={base_ce:.4f} "
                  f"info={ig:.4f} abs={abs_l:.4f} zipf={zl:.4f} "
                  f"steer={steer_norm:.4f} | "
                  f"lr={optimizer.param_groups[0]['lr']:.2e}")
            history['step'].append(global_step)
            history['loss'].append(total_loss)
            history['base_ce'].append(base_ce)
            history['info_gain'].append(ig)
            history['abs_loss'].append(abs_l)
            history['zipf_loss'].append(zl)

        del loss, out
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

print(f"\nTraining done! {global_step} steps")

In [ ]:
# ── Visualize training curves ──
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 4, figsize=(18, 4))

axes[0].plot(history['step'], history['loss'], 'b-', alpha=0.7)
axes[0].set_title('Total Loss')
axes[0].set_xlabel('Step')

axes[1].plot(history['step'], history['base_ce'], 'r-', alpha=0.7)
axes[1].set_title('Base CE (no steering)')
axes[1].set_xlabel('Step')

axes[2].plot(history['step'], history['info_gain'], 'g-', alpha=0.7)
axes[2].axhline(y=0, color='k', linestyle='--', alpha=0.3)
axes[2].set_title('Info Gain (neg=helps)')
axes[2].set_xlabel('Step')

axes[3].plot(history['step'], history['abs_loss'], 'm-', alpha=0.7)
axes[3].set_title('Abs Loss (routing prediction)')
axes[3].set_xlabel('Step')

plt.tight_layout()
plt.show()

# Show steering embedding norms + routing head
with torch.no_grad():
    w = wrapper_v9.steering_emb.weight.float()
    print(f"Steering emb norms: {[f'{n:.4f}' for n in w.norm(dim=-1).tolist()]}")
    print(f"abs_proj weight norm: {wrapper_v9.abs_proj.weight.float().norm():.4f}")

In [ ]:
# ── Probe: Inner-Monologue Diversity ─────────────────────────────────────
# For a dozen test sequences, visualize:
#   1. Per-chunk routing codes (the "inner monologue" of abstraction choices)
#   2. Code frequency histogram across all chunks
#   3. Per-sequence entropy of code distribution
#   4. 2-gram transition matrix (which code follows which)

import numpy as np
import matplotlib.pyplot as plt
from collections import Counter

N_PROBE = 12
test_ds = get_dataset(DATASET, split="test", tokenizer=tokenizer, max_length=MAX_LENGTH)
probe_indices = list(range(min(N_PROBE, len(test_ds))))

wrapper_v9.eval()
all_codes_list = []
all_texts = []

with torch.no_grad():
    for idx in probe_indices:
        sample = test_ds[idx]
        ids = sample['input_ids'].unsqueeze(0).to(DEVICE)
        attn = sample['attention_mask'].unsqueeze(0).to(DEVICE)
        wrapper_v9(ids, attn)
        codes = wrapper_v9._last_chunk_codes[0].cpu().tolist()
        all_codes_list.append(codes)
        toks = ids[0][attn[0].bool()].cpu().tolist()[:60]
        all_texts.append(tokenizer.decode(toks, skip_special_tokens=True)[:80])

wrapper_v9.train()

# ── 1. Print per-sequence code patterns ──
print("=" * 70)
print("Per-sequence routing codes (inner monologue)")
print("=" * 70)
for i, (codes, text) in enumerate(zip(all_codes_list, all_texts)):
    code_str = "".join(str(c) for c in codes)
    n_unique = len(set(codes))
    print(f"[{i:2d}] uniq={n_unique}/{C_SIZE} | {code_str[:64]}{'...' if len(code_str)>64 else ''}")
    print(f"     {text}...")

# ── 2. Global code frequency ──
flat_codes = [c for seq in all_codes_list for c in seq]
counts = Counter(flat_codes)
fig, axes = plt.subplots(1, 3, figsize=(16, 4))

bars = [counts.get(c, 0) for c in range(C_SIZE)]
colors = plt.cm.Set2(np.linspace(0, 1, C_SIZE))
axes[0].bar(range(C_SIZE), bars, color=colors)
axes[0].set_xlabel("Code")
axes[0].set_ylabel("Count")
axes[0].set_title(f"Global Code Frequency (n={len(flat_codes)})")
axes[0].set_xticks(range(C_SIZE))

# ── 3. Per-sequence entropy ──
entropies = []
for codes in all_codes_list:
    cnt = Counter(codes)
    total = len(codes)
    probs = np.array([cnt.get(c, 0) / total for c in range(C_SIZE)])
    probs = probs[probs > 0]
    ent = -np.sum(probs * np.log2(probs))
    entropies.append(ent)

axes[1].bar(range(len(entropies)), entropies, color='steelblue')
axes[1].axhline(y=np.log2(C_SIZE), color='r', linestyle='--', alpha=0.5, label=f'max={np.log2(C_SIZE):.2f}')
axes[1].set_xlabel("Sequence")
axes[1].set_ylabel("Entropy (bits)")
axes[1].set_title("Per-Sequence Code Entropy")
axes[1].legend()

# ── 4. 2-gram transition matrix ──
transition = np.zeros((C_SIZE, C_SIZE))
for codes in all_codes_list:
    for a, b in zip(codes[:-1], codes[1:]):
        transition[a][b] += 1
row_sums = transition.sum(axis=1, keepdims=True)
row_sums[row_sums == 0] = 1
trans_prob = transition / row_sums

im = axes[2].imshow(trans_prob, cmap='Blues', vmin=0, vmax=1)
axes[2].set_xlabel("Next code")
axes[2].set_ylabel("Current code")
axes[2].set_title("2-gram Transition P(next|cur)")
axes[2].set_xticks(range(C_SIZE))
axes[2].set_yticks(range(C_SIZE))
for r in range(C_SIZE):
    for c_ in range(C_SIZE):
        axes[2].text(c_, r, f"{trans_prob[r][c_]:.2f}", ha='center', va='center', fontsize=9)
fig.colorbar(im, ax=axes[2], shrink=0.8)

plt.tight_layout()
plt.show()

# ── Summary stats ──
mean_ent = np.mean(entropies)
max_ent = np.log2(C_SIZE)
global_unique = len(set(flat_codes))
print(f"\nDiversity summary:")
print(f"  Mean per-seq entropy: {mean_ent:.3f} / {max_ent:.2f} bits ({mean_ent/max_ent*100:.0f}% of max)")
print(f"  Global codes used: {global_unique}/{C_SIZE}")
print(f"  Most common code: {counts.most_common(1)[0]}")
print(f"  Least common code: {counts.most_common()[-1]}")

In [ ]:
def steer_cosine_sim(wrapper):
    """Avg pairwise cosine similarity between steering embedding vectors."""
    w = wrapper.steering_emb.weight.float()  # (C, D)
    w_norm = F.normalize(w, dim=-1)
    sim = w_norm @ w_norm.T  # (C, C)
    C = sim.size(0)
    mask = ~torch.eye(C, dtype=torch.bool, device=sim.device)
    avg_cos = sim[mask].mean().item()
    print(f"Steering emb pairwise cosine sim (C={C}):")
    print(f"  Avg off-diag: {avg_cos:.4f}")
    print(f"  Full matrix:\n{sim.cpu().numpy().round(3)}")
    return avg_cos

steer_cosine_sim(wrapper_v9)

In [ ]:
# TODO: Delete this cell (probe is in cell 5)